In [ ]:
"""
Author: Emily Vosper + Ruby Lieber
Affiliation: University of Bristol
Date: 26/06/2025

Description:
This script is used for plotting historical vs hist-nat surface tamperatures over Brazil between 2000 to 2018.
"""

In [ ]:
import os
os.environ["PROJ_DATA"] = "/user/home/al18709/breathe/.pixi/envs/default/share/proj"
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import geopandas as gpd
import pickle
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def calc_gdf_mean(gdfs):
    # 1. Stack temperature columns into a 2D array
    temps = np.stack([gdf['daily_mean_temperature'].values for gdf in gdfs], axis=0)

    # 2. Compute the mean across the first axis (ensemble members)
    mean_temp = temps.mean(axis=0)

    # 3. Copy one of the gdfs and assign the mean
    gdf_mean = gdfs[0].copy()
    gdf_mean['daily_mean_temperature'] = mean_temp

    return gdf_mean

# def process_pq(df):
    # df_grouped = df.groupby(['rgi'], observed=True, as_index=False)['daily_mean_temperature'].mean().reset_index()
    # df_merged = pd.merge(df_grouped, gdf, on='rgi', how='left')
    # df_gdf = gpd.GeoDataFrame(df_merged, geometry='geometry', crs=gdf.crs)
    # return df_gdf

def process_pq(df):
    """Group by rgi, compute mean temperature, and merge with shapefile."""
    df = df.reset_index()
    df_grouped = df.groupby('rgi', as_index=False)['daily_mean_temperature'].mean()
    df_merged = pd.merge(df_grouped, gdf, on='rgi', how='left')
    return gpd.GeoDataFrame(df_merged, geometry='geometry', crs=gdf.crs)

In [ ]:
base_path = "/bp1/geog-tropical/data/BREATHE/timeseries_igr_new"

models = {
    'ACCESS-CM2':      [f'r{r}i1p1f1' for r in [1]],
    'ACCESS-ESM1-5':   [f'r{r}i1p1f1' for r in range(1, 4)],
    'BCC-CSM2-MR':     [f'r{r}i1p1f1' for r in [1]],
    'CNRM-CM6-1':      [f'r{r}i1p1f2' for r in range(1, 4)],
    'CanESM5':         [f'r{r}i1p1f1' for r in range(1, 26) if r not in [12, 16, 17, 18, 19, 20, 21, 24]],
    'CanESM5-p2':      [f'r{r}i1p2f1' for r in range(1, 26) if r not in [3, 7, 12, 13, 14, 15, 17, 19, 23]],
    'FGOALS-g3':       [f'r{r}i1p1f1' for r in [1]],
    'GFDL-CM4':        [f'r{r}i1p1f1' for r in [1]],
    'GFDL-ESM4':       [f'r{r}i1p1f1' for r in [1]],
    'HadGEM3-GC31-LL': [f'r{r}i1p1f3' for r in range(1, 61) if r not in [6, 7, 8, 9, 10]],
    'IPSL-CM6A-LR':    [f'r{r}i1p1f1' for r in range(1, 7) if r != 3],
    'MIROC6':          [f'r{r}i1p1f1' for r in range(1, 3)],
    'MRI-ESM2-0':      [f'r{r}i1p1f1' for r in range(1, 6) if r != 4],
    'NorESM2-LM':      [f'r{r}i1p1f1' for r in range(1, 4)],
}

parquets = {}

def load_ts(path):
    df = pd.read_parquet(path)
    df = df.set_index(['date', 'rgi']).sort_index()
    return df.loc['2000':'2018']

for model, runs in models.items():
    for run in runs:
        if model == 'CanESM5-p2':
            hist_file = f"{base_path}/tas_CanESM5_historical_{run}_timeseries_2000_2020.parquet"
            nat_file  = f"{base_path}/tas_CanESM5_hist-nat_{run}_timeseries_2000_2020.parquet"
        else:
            hist_file = f"{base_path}/tas_{model}_historical_{run}_timeseries_2000_2020.parquet"
            nat_file  = f"{base_path}/tas_{model}_hist-nat_{run}_timeseries_2000_2020.parquet"

        parquets[f"{model}_hist_{run}"]     = load_ts(hist_file)
        parquets[f"{model}_hist-nat_{run}"] = load_ts(nat_file)

In [ ]:
polys = "/bp1/geog-tropical/data/BREATHE/RG2017_rgi_20180911/RG2017_rgi.shp"
gdf = gpd.read_file(polys)
gdfs = {key: process_pq(df) for key, df in parquets.items()}

In [ ]:
# group by model and experiment type
for model, runs in models.items():
    globals()[f'{model}_hist_mean'] = calc_gdf_mean([gdfs[f"{model}_hist_{run}"] for run in runs])
    globals()[f'{model}_hist_nat_mean'] = calc_gdf_mean([gdfs[f"{model}_hist-nat_{run}"] for run in runs])

# canesm_hist_mean    = calc_gdf_mean([gdfs[f"CanESM5_hist_r{r}"]     for r in range(1, 11)])
# canesm_hist_nat_mean = calc_gdf_mean([gdfs[f"CanESM5_hist-nat_r{r}"] for r in range(1, 11)])

# mri_hist_mean    = calc_gdf_mean([gdfs[f"MRI-ESM2-0_hist_r{r}"]     for r in [1, 3, 5]])
# mri_hist_nat_mean = calc_gdf_mean([gdfs[f"MRI-ESM2-0_hist-nat_r{r}"] for r in [1, 3, 5]])

# gfdl_hist_mean    = calc_gdf_mean([gdfs["GFDL-CM4_hist_r1"]])
# gfdl_hist_nat_mean = calc_gdf_mean([gdfs["GFDL-CM4_hist-nat_r1"]])

# multi-model means
hist_mean    = calc_gdf_mean([globals()[f'{model}_hist_mean'] for model in models.keys()])
hist_nat_mean = calc_gdf_mean([globals()[f'{model}_hist_nat_mean'] for model in models.keys()])

# hist_mean    = calc_gdf_mean([canesm_hist_mean, mri_hist_mean, gfdl_hist_mean])
# hist_nat_mean = calc_gdf_mean([canesm_hist_nat_mean, mri_hist_nat_mean, gfdl_hist_nat_mean])

# anomaly
anom_pq = hist_mean.copy()
anom_pq["temperature_anomaly"] = (
    hist_mean["daily_mean_temperature"] - hist_nat_mean["daily_mean_temperature"]
)

In [ ]:
def plot_brazil(ax, fig, anom_pq):
    # plot anomaly map
    anom_plot = anom_pq.plot(
        column="temperature_anomaly",
        ax=ax,
        cmap="coolwarm",
        vmin=-2,
        vmax=2,
        legend=False,
        edgecolor="#f5c4b2",
        linewidth=0.2,
    )
    ax.set_title("Mean surface temperature anomaly \n factual (ALL) vs counterfactual (NAT)", fontsize=14, pad=10)

    # clean axes
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

    # colorbar
    sm = anom_plot.collections[0]  # ScalarMappable
    cbar = fig.colorbar(sm, ax=ax, orientation="vertical", shrink=0.7, aspect=20, pad=0.01)
    cbar.set_label("Mean temperature anomaly (°C)", fontsize=12)
    cbar.outline.set_visible(False)
    cbar.ax.tick_params(labelsize=10)
    
    cities = [
    {"name": "Rio Branco",     "x": -67.809, "y":  -9.974, "dx":  0.4,  "dy": -0.6,  "marker": "o"},
    {"name": "Maceió",         "x": -35.735, "y":  -9.666, "dx":  0.4,  "dy": -0.5,  "marker": "o"},
    {"name": "Macapá",         "x": -51.050, "y":   0.034, "dx":  0.45, "dy":  0.0,  "marker": "o"},
    {"name": "Manaus",         "x": -60.021, "y":  -3.122, "dx":  0.4,  "dy": -0.75, "marker": "o"},
    {"name": "Salvador",       "x": -38.505, "y": -12.986, "dx":  0.3,  "dy": -0.6,  "marker": "o"},
    {"name": "Fortaleza",      "x": -38.529, "y":  -3.724, "dx":  0.3,  "dy":  0.3,  "marker": "o"},
    {"name": "Brasília",       "x": -47.895, "y": -15.807, "dx": -2.0,  "dy":  0.5,  "marker": "s"},
    {"name": "Vitória",        "x": -40.308, "y": -20.315, "dx":  0.4,  "dy": -0.7,  "marker": "o"},
    {"name": "Goiânia",        "x": -49.264, "y": -16.686, "dx":  0.4,  "dy": -0.7,  "marker": "o"},
    {"name": "São Luís",       "x": -44.302, "y":  -2.530, "dx":  0.3,  "dy":  0.1,  "marker": "o"},
    {"name": "Cuiabá",         "x": -56.098, "y": -15.601, "dx":  0.4,  "dy": -0.5,  "marker": "o"},
    {"name": "Campo Grande",   "x": -54.620, "y": -20.469, "dx": -4.6,  "dy": -1.0,  "marker": "o"},
    {"name": "Belo Horizonte", "x": -43.938, "y": -19.920, "dx":  0.4,  "dy":  0.1,  "marker": "o"},
    {"name": "Belém",          "x": -48.503, "y":  -1.455, "dx":  0.5,  "dy": -0.6,  "marker": "o"},
    {"name": "João Pessoa",    "x": -34.864, "y":  -7.121, "dx":  0.5,  "dy": -0.25, "marker": "o"},
    {"name": "Curitiba",       "x": -49.265, "y": -25.428, "dx":  0.5,  "dy": -0.75, "marker": "o"},
    {"name": "Recife",         "x": -34.883, "y":  -8.059, "dx":  0.4,  "dy": -0.6,  "marker": "o"},
    {"name": "Teresina",       "x": -42.803, "y":  -5.090, "dx": -2.6,  "dy": -0.7,  "marker": "o"},
    {"name": "Rio de Janeiro", "x": -43.189, "y": -22.917, "dx":  0.4,  "dy": -0.75, "marker": "o"},
    {"name": "Natal",          "x": -35.209, "y":  -5.795, "dx":  0.4,  "dy": -0.2,  "marker": "o"},
    {"name": "Porto Alegre",   "x": -51.211, "y": -30.038, "dx":  0.4,  "dy": -0.75, "marker": "o"},
    {"name": "Porto Velho",    "x": -63.903, "y":  -8.761, "dx":  0.4,  "dy": -0.6,  "marker": "o"},
    {"name": "Boa Vista",      "x": -60.673, "y":   2.823, "dx":  0.5,  "dy": -0.2,  "marker": "o"},
    {"name": "Florianópolis",  "x": -48.548, "y": -27.595, "dx":  0.4,  "dy": -0.7,  "marker": "o"},
    {"name": "São Paulo",      "x": -46.638, "y": -23.568, "dx":  0.3,  "dy": -1.0,  "marker": "o"},
    {"name": "Aracaju",        "x": -37.073, "y": -10.947, "dx":  0.4,  "dy": -0.65, "marker": "o"},
    {"name": "Palmas",         "x": -48.333, "y": -10.184, "dx":  0.4,  "dy": -0.7,  "marker": "o"},
]

    size = 10
    for city in cities:
        ax.scatter(city["x"], city["y"], color="black", s=size, marker=city["marker"])
        ax.annotate(
            city["name"],
            xy=(city["x"] + city["dx"], city["y"] + city["dy"]),
            xycoords="data",
            fontsize=9,
        )

    return ax, fig

In [ ]:
def plot_timeseries(ax):
    # Load obs and smooth
    obs_timeseries = xr.open_dataarray("/bp1/geog-tropical/data/BREATHE/attr_warming/BR_DWGD_Tmean_annual_timeseries.nc")
    obs_smoothed = obs_timeseries.rolling(time=5, center=True, min_periods=1).mean()

    # Load model timeseries
    # with open("/bp1/geog-tropical/data/BREATHE/attr_warming/plotting/timeseries.pkl", "rb") as f:
    #     timeseries = pickle.load(f)
    
    with open("/bp1/geog-tropical/data/BREATHE/attr_warming/plotting/timeseries_all_models.pkl", "rb") as f:
        timeseries = pickle.load(f)
        
        

    # Separate runs by experiment
    historical = {m: da.mean(dim='ensemble') for (m, r), da in timeseries.items() if r == 'historical'}
    hist_nat   = {m: da.mean(dim='ensemble') for (m, r), da in timeseries.items() if r == 'hist-nat'}

    # Combine into model dimension
    historical_da = xr.concat(historical.values(), dim='model').assign_coords(model=list(historical.keys()))
    hist_nat_da   = xr.concat(hist_nat.values(), dim='model').assign_coords(model=list(hist_nat.keys()))

    # Multi-model means
    historical_mmm = historical_da.mean(dim="model")
    hist_nat_mmm   = hist_nat_da.mean(dim="model")

    # Helper to compute anomalies against 1961–1990
    def anomalies(da):
        clim = da.sel(time=slice("1961", "1990")).mean()
        return da - clim

    obs_anom = anomalies(obs_smoothed)
    all_anom = anomalies(historical_mmm)
    nat_anom = anomalies(hist_nat_mmm)
    ant_anom = all_anom - nat_anom  # unused, but keeping for clarity

    # Ensemble spread (stack all members first)
    all_stack = xr.concat([anomalies(da) for (m, r), da in timeseries.items() if r == "historical"], dim="ensemble")
    nat_stack = xr.concat([anomalies(da) for (m, r), da in timeseries.items() if r == "hist-nat"], dim="ensemble")

    all_p5, all_p95 = all_stack.quantile([0.05, 0.95], dim="ensemble")
    nat_p5, nat_p95 = nat_stack.quantile([0.05, 0.95], dim="ensemble")

    # Colors
    c1, c2 = "#f9a822", "#5C9C62"

    # Plot mean lines
    all_anom.plot(ax=ax, label="ALL", color=c1, linewidth=2)
    nat_anom.plot(ax=ax, label="NAT", color=c2, linewidth=2)
    obs_anom.plot(ax=ax, label="OBS", c="black", linewidth=2)

    # Plot spreads
    ax.fill_between(all_p5.time, all_p5, all_p95, color=c1, alpha=0.3)
    ax.fill_between(nat_p5.time, nat_p5, nat_p95, color=c2, alpha=0.3)

    # Labels and legend
    ax.set_title("Mean surface temperature over Brazil \n simulated vs observations", fontsize=14)
    ax.set_ylabel("Temperature Anomaly (°C)", fontsize=12)
    ax.set_xlabel("Year", fontsize=12)
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)

    return ax

In [ ]:
def plot_scaling(ax1, ax2):
    # Colors and transparency
    warming_colors = ['#313e47', "#f9a822", "#f96635", "#5C9C62"]
    alpha = 0.8

    # Load data
    data = np.load("/bp1/geog-tropical/data/BREATHE/attr_warming/plotting/SF_CI_warming.npz")
    scaling_factors = data["scaling_factors"]
    scaling_lowers = data["scaling_lowers"]
    scaling_uppers = data["scaling_uppers"]
    warming = data["warming"]
    warming_lowers = data["warming_lowers"]
    warming_uppers = data["warming_uppers"]

    scaling_labels = ['ALL', 'ANT', 'NAT']
    warming_labels = ['OBS', 'ALL', 'ANT', 'NAT']

    # --- Left panel: Scaling Factors ---
    x = np.arange(len(scaling_factors))
    errors = [
        np.array(scaling_factors) - np.array(scaling_lowers),
        np.array(scaling_uppers) - np.array(scaling_factors)
    ]

    ax1.errorbar(
        x, scaling_factors, yerr=errors,
        fmt='o', capsize=5, color='black'
    )
    ax1.axhline(1, color='grey', linestyle='--', label="consistent with obs")
    ax1.set_xticks(x)
    ax1.set_xticklabels(scaling_labels)
    ax1.set_ylabel('Scaling Factor')
    ax1.set_title('Scaling factors (90% CI)')
    ax1.legend()

    # --- Right panel: Attributable Warming ---
    x = np.arange(len(warming))
    bars = ax2.bar(
        x, warming,
        color=warming_colors, edgecolor='black', alpha=alpha
    )

    warming_errors = [
        np.array(warming) - np.array(warming_lowers),
        np.array(warming_uppers) - np.array(warming)
    ]
    ax2.errorbar(
        x, warming, yerr=warming_errors,
        fmt='none', ecolor='black', capsize=5, elinewidth=1
    )

    ax2.set_xticks(x)
    ax2.set_xticklabels(warming_labels)
    ax2.set_ylabel('Warming (°C)')
    ax2.set_title('Attributable heating (2000–2018)')

    return ax1, ax2

In [ ]:
# Set up the figure
fig = plt.figure(figsize=(20, 16))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1.5], height_ratios=[1, 1], wspace=0.1, hspace=0.3)

# === BRAZIL MAP ===

ax1 = fig.add_subplot(gs[:, 1])
ax1,fig = plot_brazil(ax1,fig,anom_pq)
ax1.text(0, 1.14, 'd.', transform=ax1.transAxes, fontsize=18, fontweight='bold', va='top', ha='left')

# === TIMESERIES ===

ax2 = fig.add_subplot(gs[0, 0])
plot_timeseries(ax2)
ax2.text(-0.2, 1.15, 'a.', transform=ax2.transAxes, fontsize=18, fontweight='bold', va='top', ha='left')

# === SCALING FACTORS (split into two stacked axes) ===
inner_gs = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs[1, 0], hspace=0.3)

ax3a = fig.add_subplot(inner_gs[0, 0])
ax3b = fig.add_subplot(inner_gs[1, 0])

ax3a,ax3b = plot_scaling(ax3a,ax3b)  

ax3a.text(-0.2, 1.15, 'b.', transform=ax3a.transAxes, fontsize=18, fontweight='bold', va='top', ha='left')
ax3b.text(-0.2, 1.15, 'c.', transform=ax3b.transAxes, fontsize=18, fontweight='bold', va='top', ha='left')

fig.subplots_adjust(
    top=0.7,     # Lower this to add space at the top
    bottom=0.2,  # Raise this to add space at the bottom
)

plt.savefig('Figure_1.svg',dpi=300,bbox_inches='tight')

In [ ]:
fig_a, ax_a = plt.subplots(figsize=(14, 8))
plot_timeseries(ax_a)
ax_a.text(-0.2, 1.15, 'a.', transform=ax_a.transAxes, fontsize=18, fontweight='bold', va='top', ha='left')
plt.savefig('Figure_1a.svg', dpi=300, bbox_inches='tight')

In [ ]:
fig_b, ax_b = plt.subplots(figsize=(14, 8))
_, ax_c_dummy = plt.subplots(figsize=(8, 4))  # throwaway axis, plot_scaling needs both
plot_scaling(ax_b, ax_c_dummy)
plt.close(ax_c_dummy.figure)  # discard the dummy figure
ax_b.text(-0.2, 1.15, 'b.', transform=ax_b.transAxes, fontsize=18, fontweight='bold', va='top', ha='left')
fig_b.savefig('Figure_1b.svg', dpi=300, bbox_inches='tight')

In [ ]:
_, ax_b_dummy = plt.subplots(figsize=(14, 8))  # throwaway axis
fig_c, ax_c = plt.subplots(figsize=(8, 4))
plot_scaling(ax_b_dummy, ax_c)
plt.close(ax_b_dummy.figure)  # discard the dummy figure
ax_c.text(-0.2, 1.15, 'c.', transform=ax_c.transAxes, fontsize=18, fontweight='bold', va='top', ha='left')
fig_c.savefig('Figure_1c.svg', dpi=300, bbox_inches='tight')

In [ ]:
fig_d, ax_d = plt.subplots(figsize=(14, 16))
ax_d, fig_d = plot_brazil(ax_d, fig_d, anom_pq)
ax_d.text(0, 1.14, 'd.', transform=ax_d.transAxes, fontsize=18, fontweight='bold', va='top', ha='left')
plt.savefig('Figure_1d.svg', dpi=300, bbox_inches='tight')